# SAM 3 / 3.1 video segmentation for piano footage (Colab)
Official Meta SAM 3 (`facebookresearch/sam3`). Runs **two tracked passes** over the same frames:
1. **performer** (text prompt, optional click refinements)
2. **piano** (text prompt, optional click refinements)

SAM 3 allows one text concept per session, so each pass has its own session. Raw masks are written to Google Drive as 8-bit PNGs (0/255) with identical frame numbering. Refinement and export happen on the Mac.

**Before you run:** (a) request access to `facebook/sam3` and `facebook/sam3.1` on Hugging Face, (b) add a read token as Colab secret `HF_TOKEN` (key icon in left bar, enable notebook access), (c) Runtime > Change runtime type > GPU.

In [ ]:
# 1. Environment check
!nvidia-smi
import sys, torch
print("python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.version.cuda, "| gpu ok:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU runtime first"

In [ ]:
# 2. Mount Drive and set persistent locations (wheelhouse lives on Drive; HF cache stays local)
from google.colab import drive
drive.mount("/content/drive")
import os
ROOT = "/content/drive/MyDrive/piano-seg"
os.makedirs(f"{ROOT}/wheelhouse", exist_ok=True)
# NOTE: do NOT put the Hugging Face cache on Drive: Drive has no symlinks and HF's cache layout breaks (stub files). Local disk is fine (3.5 GB, ~30 s).
os.environ["HF_HOME"] = "/content/hf_local"

In [ ]:
# 3. Install official SAM 3 with a Drive wheelhouse so new runtimes skip the slow builds.
# sam3 pins numpy>=1.26,<2 and lists Python <=3.12; Colab is newer, so numpy 1.26 compiles from source (slow, ONCE: cached below).
# opencv 5 needs numpy>=2, so we pin opencv 4.10 (numpy 1.x compatible).
import sys, subprocess, glob
def sh(c): subprocess.check_call(c, shell=True)
SAM3_REF = "main"   # TODO: pin to a commit hash once a good run is confirmed
tag = f"py{sys.version_info.major}{sys.version_info.minor}"
WH = f"{ROOT}/wheelhouse/{tag}"; os.makedirs(WH, exist_ok=True)
import numpy; numpy_before = numpy.__version__
if not (glob.glob(f"{WH}/numpy-1.26.4*.whl") and glob.glob(f"{WH}/sam3-*.whl")):
    sh(f"pip wheel -q --no-deps -w {WH} numpy==1.26.4 ftfy==6.1.1 iopath git+https://github.com/facebookresearch/sam3.git@{SAM3_REF}")
sh(f"pip install -q --find-links {WH} numpy==1.26.4 opencv-python-headless==4.10.0.84 matplotlib scikit-learn einops ftfy==6.1.1 iopath sam3")
sh("pip freeze > " + f"{ROOT}/requirements.lock")   # record exactly what worked
if numpy_before != "1.26.4":
    print("numpy was swapped under the running kernel; restarting runtime. After it restarts, run cell 2 (mount) then cell 3 again, then continue.")
    os.kill(os.getpid(), 9)
import sam3, cv2, numpy as np; print("sam3 at", sam3.__file__, "| numpy", np.__version__, "| cv2", cv2.__version__)

In [ ]:
# 3. Hugging Face auth (token is read from Colab Secrets; never printed)
from google.colab import userdata
from huggingface_hub import login, whoami
login(token=userdata.get("HF_TOKEN"))
print("logged in as:", whoami()["name"])
# Gated-repo access is verified when the checkpoint downloads in step 6; a 403 there means access wasn't approved yet.

In [ ]:
# 4. Config
CONFIG = dict(
    # Either an MP4/MOV on Drive (frames get extracted) ...
    video_path="/content/drive/MyDrive/PianoClaude/TujhseNarazPiano.mov",
    # ... outputs go here:
    out_root="/content/drive/MyDrive/piano-seg/outputs",
    model="sam3.1",             # "sam3.1" (Object Multiplex) or "sam3"
    max_side=1920,             # inference frame long side (original aspect kept)
    max_seconds=10,
    chunk=24, overlap=4,        # SAM 3.1 keeps per-frame features on the GPU (~0.5 GB/frame at 1080x1920): process in overlapping chunks            # None = whole video. Keep 10 for the first test.
    passes={
        "performer": dict(text="person playing piano", keep_obj_ids=None,
                          clicks=[]),   # clicks: dict(frame=0, obj_id=1, points=[[x,y],..], labels=[1,0,..]) in pixels of the inference frames
        "piano":     dict(text="grand piano", keep_obj_ids=None, clicks=[]),
    },
)
import os, json, subprocess
clip = os.path.splitext(os.path.basename(CONFIG["video_path"]))[0]
OUT = f'{CONFIG["out_root"]}/{clip}'
FRAMES = f"/content/frames/{clip}"
os.makedirs(OUT, exist_ok=True); os.makedirs(FRAMES, exist_ok=True)

In [ ]:
# 5. Extract frames with FFmpeg (JPEG q=2, aspect preserved, no upscaling) + probe metadata
import shutil
p = json.loads(subprocess.check_output(["ffprobe","-v","error","-select_streams","v:0","-show_entries",
    "stream=width,height,r_frame_rate,nb_frames,duration,codec_name,pix_fmt:stream_tags=rotate:stream_side_data=rotation",
    "-of","json",CONFIG["video_path"]]))["streams"][0]
print(p)
ms = CONFIG["max_side"]
vf = f"scale='if(gt(iw,ih),min({ms},iw),-2)':'if(gt(iw,ih),-2,min({ms},ih))':flags=lanczos"
cmd = ["ffmpeg","-y","-v","error","-i",CONFIG["video_path"]]
if CONFIG["max_seconds"]: cmd += ["-t",str(CONFIG["max_seconds"])]
cmd += ["-vf",vf,"-fps_mode","passthrough","-qscale:v","2",f"{FRAMES}/%05d.jpg"]
if not os.listdir(FRAMES): subprocess.check_call(cmd)
frames = sorted(os.listdir(FRAMES)); print(len(frames), "frames")
from PIL import Image
W,H = Image.open(f"{FRAMES}/{frames[0]}").size
meta = dict(source=CONFIG["video_path"], r_frame_rate=p["r_frame_rate"], src_width=p["width"], src_height=p["height"],
            infer_width=W, infer_height=H, n_frames=len(frames), model=CONFIG["model"])
json.dump(meta, open(f"{OUT}/meta.json","w"), indent=2); print(meta)

In [ ]:
# 6. Build predictor (SAM 3.1 multiplex or SAM 3)
# Workarounds for current sam3 main (verified against the official source):
#  - multiplex builders default use_fa3=True (FlashAttention-3 = Hopper/H100 only) -> use_fa3=False on L4/T4/A100
#  - Sam3BasePredictor.start_session always passes offload_state_to_cpu, but the multiplex model's init_state() rejects it -> drop it
import torch, numpy as np, cv2
from sam3.model_builder import build_sam3_video_predictor
try:
    from sam3.model_builder import build_sam3_multiplex_video_predictor
except ImportError:
    build_sam3_multiplex_video_predictor = None
if CONFIG["model"] == "sam3.1":
    assert build_sam3_multiplex_video_predictor, "Installed sam3 lacks 3.1; reinstall from git main"
    predictor = build_sam3_multiplex_video_predictor(use_fa3=False)
    predictor.model.init_state = (lambda f: (lambda *a, offload_state_to_cpu=None, **k: f(*a, **k)))(predictor.model.init_state)
else:
    predictor = build_sam3_video_predictor()
print("predictor ready |", torch.cuda.memory_allocated()/1e9, "GB allocated")

In [ ]:
# 7. Helpers
import matplotlib.pyplot as plt
def req(**kw): return predictor.handle_request(request=kw)

def start(path=None): return req(type="start_session", resource_path=path or FRAMES, offload_video_to_cpu=True)["session_id"]

def add_text(sid, text, frame=0):
    return req(type="add_prompt", session_id=sid, frame_index=frame, text=text)["outputs"]

def add_clicks(sid, c):
    pts = torch.tensor([[x/W, y/H] for x,y in c["points"]], dtype=torch.float32)
    lab = torch.tensor(c["labels"], dtype=torch.int32)
    return req(type="add_prompt", session_id=sid, frame_index=c["frame"], points=pts, point_labels=lab, obj_id=c["obj_id"])["outputs"]

def show(frame, out, title):
    img = cv2.cvtColor(cv2.imread(f"{FRAMES}/{frames[frame]}"), cv2.COLOR_BGR2RGB).copy()
    rng = np.random.default_rng(0)
    for i, oid in enumerate(out["out_obj_ids"].tolist()):
        m = out["out_binary_masks"][i].astype(bool)
        if m.any():
            col = rng.integers(60,255,3); img[m] = (0.5*img[m] + 0.5*col).astype(np.uint8)
            ys,xs = np.where(m)
            cv2.putText(img, f"id {oid}", (int(xs.mean()), int(ys.mean())), cv2.FONT_HERSHEY_SIMPLEX, 1.4, (255,255,255), 3)
    plt.figure(figsize=(5,8)); plt.imshow(img); plt.title(title); plt.axis("off"); plt.show()

import shutil, gc
def free_gpu(): gc.collect(); torch.cuda.empty_cache()

def run_chunked(name):
    # Track one concept over the clip in overlapping chunks; later chunk wins on overlap; overlap IoU is recorded as a stability check.
    cfg = CONFIG["passes"][name]; C, OV = CONFIG["chunk"], CONFIG["overlap"]
    n = len(frames); d = f"{OUT}/raw_masks/{name}"; os.makedirs(d, exist_ok=True)
    stats, per_obj, bnd, prev = {}, {}, [], {}
    torch.cuda.reset_peak_memory_stats(); s0 = 0
    while s0 < n:
        e = min(s0 + C, n); cd = f"/content/chunks/{name}"
        shutil.rmtree(cd, ignore_errors=True); os.makedirs(cd)
        for i in range(s0, e): os.symlink(f"{FRAMES}/{frames[i]}", f"{cd}/{i-s0:05d}.jpg")
        sid = start(cd)
        try:
            add_text(sid, cfg["text"], 0)
            for c in cfg["clicks"]:
                if s0 <= c["frame"] < e: add_clicks(sid, dict(c, frame=c["frame"]-s0))
            cur = {}
            for r in predictor.handle_stream_request(request=dict(type="propagate_in_video", session_id=sid)):
                gi, o = s0 + r["frame_index"], r["outputs"]
                union = np.zeros((H,W), bool)
                for i, oid in enumerate(o["out_obj_ids"].tolist()):
                    m = o["out_binary_masks"][i].astype(bool)
                    per_obj.setdefault(oid, {})[gi] = int(m.sum())
                    if cfg["keep_obj_ids"] is None or oid in cfg["keep_obj_ids"]: union |= m
                if gi in prev:
                    inter = (union & prev[gi]).sum(); uni = (union | prev[gi]).sum()
                    bnd.append((gi, float(inter/uni) if uni else 1.0))
                cv2.imwrite(f"{d}/{gi:05d}.png", union.astype(np.uint8)*255)
                stats[gi] = int(union.sum()); cur[gi] = union
        finally:
            req(type="close_session", session_id=sid); free_gpu()
        prev = {g: m for g, m in cur.items() if g >= e - OV}
        print(f"{name}: frames {s0}-{e-1} done | peak {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
        if e >= n: break
        s0 = e - OV
    json.dump(dict(area_px=stats, per_object_area_px=per_obj, boundary_iou=bnd, size=[W,H], chunk=C, overlap=OV,
                   peak_gpu_gb=torch.cuda.max_memory_allocated()/1e9), open(f"{OUT}/raw_masks/{name}_stats.json","w"))
    if bnd: print("overlap IoU min/mean:", min(b for _,b in bnd), sum(b for _,b in bnd)/len(bnd))
    return stats

## 8. Pass A: performer. 8a previews the prompt on frame 0 (check ids/mask), 8b tracks the whole clip in chunks.

In [ ]:
cfg = CONFIG["passes"]["performer"]
sid = start(); out = add_text(sid, cfg["text"], 0)
for c in cfg["clicks"]: out = add_clicks(sid, c)
show(0, out, f'performer: "{cfg["text"]}"  (check ids)'); print("ids", out["out_obj_ids"], "areas", [int(m.sum()) for m in out["out_binary_masks"]])
req(type="close_session", session_id=sid); free_gpu()

In [ ]:
st = run_chunked("performer"); print(len(st), "frames saved")

## 9. Pass B: piano (same procedure)

In [ ]:
cfg = CONFIG["passes"]["piano"]
sid = start(); out = add_text(sid, cfg["text"], 0)
for c in cfg["clicks"]: out = add_clicks(sid, c)
show(0, out, f'piano: "{cfg["text"]}"  (check ids)'); print("ids", out["out_obj_ids"], "areas", [int(m.sum()) for m in out["out_binary_masks"]])
req(type="close_session", session_id=sid); free_gpu()

In [ ]:
st = run_chunked("piano"); print(len(st), "frames saved")

In [ ]:
# 10. Quick sanity montage of saved masks (frames 0, middle, last)
import glob
for name in ("performer","piano"):
    fs = sorted(glob.glob(f"{OUT}/raw_masks/{name}/*.png")); idx = [0, len(fs)//2, len(fs)-1]
    fig, ax = plt.subplots(1,3,figsize=(9,6))
    for a,i in zip(ax,idx): a.imshow(cv2.imread(fs[i],0), cmap="gray"); a.set_title(f"{name} {i}"); a.axis("off")
    plt.show()
print("Done. Download/sync:", OUT)